# 第7章 NMF をゼロから実装して、AuxIVA・ILRMA で音源を分離する（ノートブック版）

教科書の第7章の 7-6 のプログラムを、1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約 3 分（分離の実験を試しの設定にしたとき）。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | NMF（KL と IS）で分解し、成分を聞く | C の 4・5 | `nmf.py` |
| 2 | IS の更新の平方根を外して比べる | C の 6 | `nmf.py` の `nmf()` |
| 3 | 部屋のシミュレーションで混ぜ、AuxIVA と ILRMA で分離する（**ノートブックでは 4 試行の試しの設定**） | D の 7・8 | `bss_experiment.py`・`bss.py` |
| 4 | ILRMA と AuxIVA のコストが増えないことを確かめる | D の 9 | `bss.py` |
| 5 | 統計的に比べる | E の 10 | `bss_stats.py` |

- 次の 4 つのセルが、7-6 の `nmf.py`・`bss.py`・`bss_experiment.py`・`bss_stats.py` を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の 7-6 と一字も変えていません。
- 教科書でターミナルに打つ `uv run python nmf.py …` などは、ノートブックでは `run_py("nmf.py", …)` で実行します（同じ Python でプログラムを動かし、出力を表示します。日本語のフォントの候補のうち見つからないものの知らせは省きます）。図は保存された PNG をセルの下に表示し、`ok()` で教科書の手順の「確かめること」を確かめます。
- **入れていないもの**：手順 A の `setup_ch7.sh`（`uv add` とファイルの保存。ここでは 0 のセルと `%%writefile` が代わり）、**手順 B の録音（`record.py`）**。録音には Mac のマイク（と `sounddevice`）が要り、Colab からは使えないので、このノートブックには入れていません。代わりに、教科書の `run_ch7.sh` の「録音がなければ」の行と同じく、librosa の例の音源（NMF は `trumpet`、分離は `libri1`（話し声）と `trumpet`）を使います（初回はダウンロードします。ネットワークが要ります）。自分の録音で試すときは、wav をこのノートブックの隣にアップロードし、`run_py` の引数を `run_ch7.sh` のとおりに書きかえてください。
- **3 節の分離の実験は重い**ので、ノートブックでは試しの設定（`--trials 4`：残響時間 2 × 4 試行 × 4 手法）を既定にしています。教科書の本番（`--trials 14`、28 試行。執筆時の Mac で約 7 分、Colab ではその数倍）にするときは、セルの `TRIALS = 4` をコメントのとおりに書きかえてください。4 試行では、5 節の統計（後半の 2 試行で比べる）でウィルコクソンの検定は「6試行未満なので検定は省略」と表示され、教科書の 7-7 の数値とは合いません。ノートブックの確かめは、試行の数に関係なく成り立つことだけにしてあります。
- **音を聞くときの注意**：再生ボタンは `listen_files()` が作ります。分離音の wav は最大 0.9 で保存されているので、そのままでは大きすぎます。`listen_files()` は、並べて聞くファイルの中で一番大きい値が 0.3（約 −10 dBFS）になるように**全部を同じ倍率で小さくしてから**、`IPython.display.Audio(..., normalize=False)` で鳴らします。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を一番小さくしてください。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、pyroomacoustics・librosa・scipy の版で変わることがあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile"),
                 ("pyroomacoustics", "pyroomacoustics"), ("fast_bss_eval", "fast_bss_eval")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, pandas, librosa, pyroomacoustics, fast_bss_eval
from importlib.metadata import version
print("numpy", numpy.__version__, "scipy", scipy.__version__, "pandas", pandas.__version__,
      "librosa", librosa.__version__, "pyroomacoustics", pyroomacoustics.__version__,
      "fast_bss_eval", version("fast_bss_eval"))
print("（教科書は librosa 1.0.1・pyroomacoustics 0.10.1・fast_bss_eval 0.1.4 で確かめた。版をノートに書く：手順 A）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の 4 つのセルで、7-6 のプログラムを書き出します（録音の `record.py` は使わないので書き出しません）。

In [ ]:
%%writefile nmf.py
#!/usr/bin/env python3
"""第7章：NMF（非負値行列因子分解）を NumPy でゼロから実装する

使い方: cd ~/home-lab/ch7 && uv run python nmf.py data/my_voice.wav --div is --k 8 --iter 300
  --div : kl（一般化 KL ダイバージェンス。振幅スペクトログラムに使う）
          is（板倉–斎藤ダイバージェンス。パワースペクトログラムに使う）
  --k   : 基底の数
入力の wav が無いときは --example trumpet のように librosa の例の音源も使える（ダウンロードが必要）

出力: nmf_<名前>_<div>.png（コストの減り方・基底・アクティベーション）、
      nmf_<名前>_<div>_k??.wav（基底ごとに分けた音。ウィーナーフィルタ風のマスクで作る）
"""
import argparse
import pathlib

import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
EPS = 1e-12


def cost_kl(X, Y):
    """一般化 KL ダイバージェンス D(X|Y) = Σ (X log(X/Y) − X + Y)"""
    return float(np.sum(X * np.log((X + EPS) / (Y + EPS)) - X + Y))


def cost_is(X, Y):
    """板倉–斎藤ダイバージェンス D(X|Y) = Σ (X/Y − log(X/Y) − 1)"""
    r = (X + EPS) / (Y + EPS)
    return float(np.sum(r - np.log(r) - 1))


def nmf(X, k, div="kl", n_iter=200, seed=0):
    """X（F×T、非負）を W（F×k）H（k×T）に分解する。補助関数法による乗法更新。

    KL：W ← W·[(X/Y)Hᵀ]/[1Hᵀ]、H ← H·[Wᵀ(X/Y)]/[Wᵀ1]
    IS：W ← W·√{[(X/Y²)Hᵀ]/[(1/Y)Hᵀ]}、H ← H·√{[Wᵀ(X/Y²)]/[Wᵀ(1/Y)]}
    （IS の平方根は、補助関数法で導くと出てくるもの。これでコストが増えないことが保証される）
    戻り値: W, H, コストの履歴
    """
    rng = np.random.default_rng(seed)
    F, T = X.shape
    W = rng.uniform(0.1, 1.0, (F, k)) * X.mean()
    H = rng.uniform(0.1, 1.0, (k, T))
    cost = cost_kl if div == "kl" else cost_is
    hist = []
    for _ in range(n_iter):
        Y = W @ H + EPS
        if div == "kl":
            W *= ((X / Y) @ H.T) / (H.sum(axis=1)[None, :] + EPS)
            Y = W @ H + EPS
            H *= (W.T @ (X / Y)) / (W.sum(axis=0)[:, None] + EPS)
        else:
            W *= np.sqrt(((X / Y ** 2) @ H.T) / ((1 / Y) @ H.T + EPS))
            Y = W @ H + EPS
            H *= np.sqrt((W.T @ (X / Y ** 2)) / (W.T @ (1 / Y) + EPS))
        s = W.sum(axis=0) + EPS                     # 基底の和を1にそろえ、その分を H に移す（積は変わらない）
        W /= s
        H *= s[:, None]
        hist.append(cost(X, W @ H + EPS))
    return W, H, np.array(hist)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("wav", nargs="?", default=None)
    ap.add_argument("--example", default=None, help="librosa の例の音源の名前（trumpet など）")
    ap.add_argument("--div", choices=["kl", "is"], default="kl")
    ap.add_argument("--k", type=int, default=8)
    ap.add_argument("--iter", type=int, default=300)
    ap.add_argument("--sr", type=int, default=16000)
    ap.add_argument("--nfft", type=int, default=1024)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    path = a.wav or librosa.ex(a.example or "trumpet")
    name = pathlib.Path(a.wav).stem if a.wav else (a.example or "trumpet")
    y, sr = librosa.load(path, sr=a.sr, mono=True)
    hop = a.nfft // 4
    S = librosa.stft(y, n_fft=a.nfft, hop_length=hop)
    X = np.abs(S) if a.div == "kl" else np.abs(S) ** 2       # KL は振幅、IS はパワー
    print(f"{name}：{len(y)/sr:.1f} 秒、スペクトログラム {X.shape[0]}×{X.shape[1]}、{a.div.upper()}、k = {a.k}")
    W, H, hist = nmf(X, a.k, a.div, a.iter, a.seed)
    inc = np.max(np.diff(hist) / np.abs(hist[:-1]))
    print(f"コスト：最初 {hist[0]:.4g} → 最後 {hist[-1]:.4g}。1回の更新で増えた割合の最大 {inc:.2e}"
          f"（{'増えていない：補助関数法のとおり' if inc < 1e-9 else '増えた回がある：実装を見直す'}）")

    # 基底ごとの音：パワーの割合をマスクにして元の複素スペクトログラムに掛ける（位相は元のまま）
    P = (W[:, :, None] * H[None, :, :]) ** (2 if a.div == "kl" else 1)
    mask = P / (P.sum(axis=1, keepdims=True) + EPS)
    order = np.argsort(-np.sum(W * H.sum(axis=1)[None, :], axis=0))   # 大きい成分から
    for rank, k in enumerate(order[:4]):
        yk = librosa.istft(mask[:, k, :] * S, hop_length=hop, length=len(y))
        sf.write(f"nmf_{name}_{a.div}_k{rank:02d}.wav", yk, sr)
    print(f"大きい成分の上位4つを nmf_{name}_{a.div}_k00〜k03.wav に保存しました（聞き比べる）")

    fig = plt.figure(figsize=(12, 7))
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.semilogy(hist)
    ax1.set(xlabel="反復回数", ylabel="コスト", title=f"{a.div.upper()} ダイバージェンスの減り方")
    ax2 = fig.add_subplot(2, 2, 2)
    librosa.display.specshow(librosa.amplitude_to_db(np.abs(S), ref=np.max), sr=sr, hop_length=hop,
                             y_axis="hz", x_axis="time", ax=ax2)
    ax2.set_title("元のスペクトログラム（dB）")
    ax3 = fig.add_subplot(2, 2, 3)
    f = librosa.fft_frequencies(sr=sr, n_fft=a.nfft)
    for i, k in enumerate(order):
        ax3.plot(f, W[:, k] / W[:, k].max() + i, lw=0.8)
    ax3.set(xlabel="周波数 [Hz]", ylabel="基底（大きい順に上へずらした）", title="基底 W（各列のスペクトル）", xlim=(0, 4000))
    ax4 = fig.add_subplot(2, 2, 4)
    t = librosa.frames_to_time(np.arange(H.shape[1]), sr=sr, hop_length=hop)
    for i, k in enumerate(order):
        ax4.plot(t, H[k] / (H[k].max() + EPS) + i, lw=0.8)
    ax4.set(xlabel="時間 [s]", ylabel="アクティベーション（ずらした）", title="アクティベーション H（各行の時間変化）")
    for x in (ax1, ax3, ax4):
        x.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"nmf_{name}_{a.div}.png", dpi=120)
    print(f"nmf_{name}_{a.div}.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bss.py
#!/usr/bin/env python3
"""第7章：AuxIVA と ILRMA を NumPy でゼロから実装する（決定系：音源の数 N ＝ マイクの数 M）

記号（論文の周波数・時間の番号 i, j を、この章では f, t と書く。T はフレームの数で、
      論文の基底行列 T_n（ここでは Tb）とは別のもの）
  X : 観測の STFT。形は (F, T, M)。x_ft = [x_ft1, …, x_ftM]ᵀ
  W : 分離行列。形は (F, N, M)。y_ft = W_f x_ft。W_f の n 行目が w_nfᴴ
  Y : 分離信号 (F, T, N)、P = |Y|²
  r : 音源 n の分散（音源モデル）。AuxIVA は r_nt = √(Σ_f |y_nft|²)（周波数をまとめた大きさ）、
      ILRMA は r_nft = (T_n V_n)_ft（音源ごとの NMF）
更新（反復射影法 IP）
  U_nf = (1/T) Σ_t x_ft x_ftᴴ / r_n(ft)     （AuxIVA では φ(r)=1/r を掛ける）
  w_nf ← (W_f U_nf)⁻¹ e_n、 w_nf ← w_nf / √(w_nfᴴ U_nf w_nf)
コスト（負の対数尤度。反復で増えないことを確かめる）
  ILRMA : Σ_n Σ_ft (|y_nft|²/r_nft + log r_nft) − 2T Σ_f log|det W_f|
  AuxIVA: Σ_n Σ_t r_nt − T Σ_f log|det W_f|      （ラプラス分布の音源モデル）
          log|det| の係数が ILRMA の半分なのは、尺度を変えたラプラス分布の負の対数尤度を2で割ったものだから。
          定数倍なので、反復で増えないことの確認には影響しない
"""
import numpy as np

EPS = 1e-10


def demix(W, X):
    """Y = W X（周波数ごと）"""
    return np.einsum("fnm,ftm->ftn", W, X)


def ip_update(W, X, phi, n):
    """反復射影法で W の n 行目を更新する。phi は (F, T) の重み（1/r）"""
    F, T, M = X.shape
    U = np.einsum("ftm,ftk,ft->fmk", X, X.conj(), phi) / T          # (F, M, M)
    WU = W @ U
    e = np.zeros((F, M, 1), dtype=complex)
    e[:, n, 0] = 1.0
    w = np.linalg.solve(WU, e)[:, :, 0]                               # (F, M)
    denom = np.sqrt(np.real(np.einsum("fm,fmk,fk->f", w.conj(), U, w)))
    w = w / (denom[:, None] + EPS)
    W[:, n, :] = w.conj()
    return W


def logdet(W):
    return np.sum(np.log(np.abs(np.linalg.det(W)) + EPS))


def projection_back(Y, W, ref=0):
    """スケールのあいまいさを、基準マイク ref で観測した大きさに合わせて直す"""
    A = np.linalg.inv(W)                                              # (F, M, N) 混合行列の推定
    return Y * A[:, ref, :][:, None, :]


def auxiva(X, n_iter=100, return_cost=False):
    F, T, M = X.shape
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = []
    for _ in range(n_iter):
        for n in range(M):
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y[:, :, n]) ** 2, axis=0)) + EPS   # (T,)
            W = ip_update(W, X, np.broadcast_to(1.0 / r, (F, T)), n)
        if return_cost:
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y) ** 2, axis=0)) + EPS           # (T, N)
            hist.append(float(np.sum(r) - T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)


def ilrma(X, n_basis=4, n_iter=100, seed=0, return_cost=False):
    F, T, M = X.shape
    rng = np.random.default_rng(seed)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    Tb = rng.uniform(0.1, 1.0, (M, F, n_basis))                       # 基底 T_n
    Vb = rng.uniform(0.1, 1.0, (M, n_basis, T))                       # アクティベーション V_n
    R = Tb @ Vb                                                       # (N, F, T)
    hist = []
    for _ in range(n_iter):
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)               # (N, F, T)
        # 音源モデルの更新（IS ダイバージェンスの NMF。補助関数法の乗法更新）
        Tb *= np.sqrt(((P / R ** 2) @ np.swapaxes(Vb, 1, 2)) / ((1 / R) @ np.swapaxes(Vb, 1, 2) + EPS))
        R = Tb @ Vb + EPS
        Vb *= np.sqrt((np.swapaxes(Tb, 1, 2) @ (P / R ** 2)) / (np.swapaxes(Tb, 1, 2) @ (1 / R) + EPS))
        R = Tb @ Vb + EPS
        # 分離行列の更新（IP）
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
        # スケールの正規化（W の各行と、音源モデルの大きさをそろえる。結果は変わらない）
        Y = demix(W, X)
        lam = np.sqrt(np.mean(np.abs(Y) ** 2, axis=(0, 1))) + EPS     # (N,)
        W /= lam[None, :, None]
        R /= lam[:, None, None] ** 2
        Tb /= lam[:, None, None] ** 2
        if return_cost:
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            hist.append(float(np.sum(P / R + np.log(R)) - 2 * T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)

In [ ]:
%%writefile bss_experiment.py
#!/usr/bin/env python3
"""第7章：部屋のシミュレーションで2つの音源を2本のマイクで録った音を作り、AuxIVA と ILRMA で分離して評価する

使い方: cd ~/home-lab/ch7 && uv run python bss_experiment.py --src1 data/my_voice.wav --src2 data/my_guitar.wav \
          --rt60 0.15 0.3 --trials 14 --out results.csv
  音源を指定しないときは librosa の例（libri1 と trumpet）を使う（ダウンロードが必要）
  --rt60   : 残響時間 [s]（いくつでも）
  --trials : 残響時間ごとのくり返し（毎回、部屋の大きさと音源・マイクの位置を乱数で変える）。
             bss_stats.py は後半の半分の試行で検定し、2つの残響時間の検定をホルム法で補正するので
             14 以上（後半7試行）、できれば 20
  --ks     : ILRMA の基底の数（いくつでも）

結果は1回（試行）×手法ごとに1行：rt60, trial, method, sdr_in, sdr, sir, sar, sdr_imp（音源2つの平均）
評価は BSS Eval（fast_bss_eval）。基準は、各音源を基準マイク（0番）で録った音（残響を含む）。
各残響時間の試行 0 の混合音と分離音を sep/ に wav で保存する（聞き比べる）。
"""
import argparse
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import librosa
import pyroomacoustics as pra
import fast_bss_eval
import soundfile as sf

import bss

NFFT, HOP = 2048, 512                     # 16kHz で 128ms の窓。残響が長いほど長い窓がよいとされる


def load_sources(a, fs):
    p1 = a.src1 or librosa.ex("libri1")
    p2 = a.src2 or librosa.ex("trumpet")
    s = [librosa.load(p, sr=fs, mono=True, duration=a.dur)[0] for p in (p1, p2)]
    n = min(len(x) for x in s)
    s = np.stack([x[:n] / (np.std(x[:n]) + 1e-9) for x in s])  # 大きさをそろえる
    return s


def simulate(src, fs, rt60, rng):
    dim = np.array([rng.uniform(5, 8), rng.uniform(4, 6), 3.0])
    e_abs, max_order = pra.inverse_sabine(rt60, dim)
    room = pra.ShoeBox(dim, fs=fs, materials=pra.Material(e_abs), max_order=max_order)
    # 高さはちょうど真ん中（1.5m）を避ける（床と天井の鏡像が重なって DRR が低く出るため。7-7）
    center = np.array([dim[0] / 2 + rng.uniform(-0.5, 0.5), dim[1] / 2 + rng.uniform(-0.5, 0.5), 1.2])
    d = 0.04                                                      # マイクの間隔 4cm
    room.add_microphone_array(np.c_[center - [d / 2, 0, 0], center + [d / 2, 0, 0]])
    ang = rng.uniform(0, np.pi, 2)
    while abs(ang[0] - ang[1]) < np.radians(30):                 # 2つの音源の方向が近すぎないように
        ang = rng.uniform(0, np.pi, 2)
    for k in range(2):
        pos = center + 1.5 * np.array([np.cos(ang[k]), np.sin(ang[k]), 0])
        room.add_source(pos, signal=src[k])
    pre = room.simulate(return_premix=True)                       # (音源, マイク, 時間)
    L = src.shape[1]
    return pre.sum(axis=0)[:, :L], pre[:, 0, :L]


def evaluate(ref, est):
    sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return float(np.mean(sdr)), float(np.mean(sir)), float(np.mean(sar))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src1")
    ap.add_argument("--src2")
    ap.add_argument("--dur", type=float, default=6.0)
    ap.add_argument("--fs", type=int, default=16000)
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.15, 0.3])
    ap.add_argument("--trials", type=int, default=14)
    ap.add_argument("--ks", type=int, nargs="+", default=[2, 4, 10])
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--out", default="results.csv")
    a = ap.parse_args()
    if pathlib.Path(a.out).exists():
        sys.exit(f"{a.out} はすでにあります（上書きしないように止めました）")
    src = load_sources(a, a.fs)
    rows = []
    for rt60 in a.rt60:
        for trial in range(a.trials):
            rng = np.random.default_rng(1000 * trial + int(rt60 * 1000))
            mix, ref = simulate(src, a.fs, rt60, rng)
            L = ref.shape[1]
            X = np.stack([librosa.stft(m, n_fft=NFFT, hop_length=HOP) for m in mix], axis=-1)   # (F, T, M)
            with np.errstate(divide="ignore"):                    # 混合音そのままだと SAR が無限大になり、
                sdr_in = evaluate(ref, np.stack([mix[0], mix[0]]))[0]   # log10(0 割り) の警告が出るので止める
            if trial == 0:
                pathlib.Path("sep").mkdir(exist_ok=True)
                sf.write(f"sep/rt{int(rt60*1000)}_mix_mic0.wav", mix[0] / np.max(np.abs(mix)) * 0.9, a.fs)
            methods = {"AuxIVA": lambda: bss.auxiva(X, a.iter)[0]}
            for k in a.ks:
                methods[f"ILRMA_K{k}"] = (lambda k=k: bss.ilrma(X, k, a.iter, seed=trial)[0])
            for name, fn in methods.items():
                t0 = time.time()
                Y = fn()
                y = np.stack([librosa.istft(Y[:, :, n], hop_length=HOP, length=L) for n in range(Y.shape[2])])
                sdr, sir, sar = evaluate(ref, y)
                if trial == 0:
                    for n in range(y.shape[0]):
                        sf.write(f"sep/rt{int(rt60*1000)}_{name}_y{n+1}.wav", y[n] / (np.max(np.abs(y[n])) + 1e-9) * 0.9, a.fs)
                rows.append(dict(rt60=rt60, trial=trial, method=name, sdr_in=sdr_in, sdr=sdr, sir=sir, sar=sar,
                                 sdr_imp=sdr - sdr_in, sec=time.time() - t0))
                print(f"RT60 {rt60:.2f}s 試行 {trial}  {name:10s} SDR {sdr:6.2f} dB（改善 {sdr-sdr_in:+6.2f}）"
                      f" SIR {sir:6.2f}  SAR {sar:6.2f}  {time.time()-t0:5.1f} 秒")
            pd.DataFrame(rows).to_csv(a.out, index=False)           # 途中でも保存する
    print(f"{a.out} に保存しました（{len(rows)} 行）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bss_stats.py
#!/usr/bin/env python3
"""第7章：分離の評価結果（results.csv）を統計的に比べる

使い方: cd ~/home-lab/ch7 && uv run python bss_stats.py results.csv

やること
  1. 手法 × 残響時間ごとの SDR 改善量の平均と、試行を単位にしたブートストラップ 95% 区間
  2. ILRMA の基底の数 K の選び方と、楽観的なかたより
     試行を半分に分け、前半で一番よかった K を選ぶ。その同じ K の「前半（選ぶのに使った試行）での成績」と
     「後半（使っていない試行）での成績」を比べる。前半での成績を報告するのが【誤り】、後半での成績が【正しい】。
     1回の分け方では逆に出ることもあるので、分け方を乱数で変えて、かたより（前半 − 後半）の平均を出す
  3. 選んだ ILRMA と AuxIVA の対応のある比較（後半の試行、同じ試行どうしの差）：
     ウィルコクソンの符号順位検定（2つの残響時間の2回の検定をホルム法で補正）と、差の平均の区間
     独立な単位は「試行」。1つの試行の2つの音源の値は平均してから使う（bss_experiment.py で平均済み）
"""
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
RNG = np.random.default_rng(0)
N_SPLIT = 200                     # 分け方を変える回数（それぞれ前半・後半を入れ替えた分も使うので 400 通り）


def boot_ci(x, n=5000):
    x = np.asarray(x)
    m = [RNG.choice(x, len(x)).mean() for _ in range(n)]
    return np.percentile(m, [2.5, 97.5])


def t_ci(x):
    x = np.asarray(x)
    h = stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return x.mean() - h, x.mean() + h


def select_k(tab, tune, test):
    """前半 tune で一番よい K を選び、その K の前半での平均と後半での平均を返す"""
    m_tune = tab.loc[tune].mean()
    k = m_tune.idxmax()
    return k, m_tune[k], tab.loc[test, k].mean()


def holm(pvals):
    """ホルム法で補正した p 値（小さい順に (m − i) 倍し、単調になるようにそろえる）"""
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    for i, j in enumerate(order):
        run = max(run, (len(p) - i) * p[j])
        adj[j] = min(1.0, run)
    return adj


def main(path):
    d = pd.read_csv(path)
    print("[1] SDR 改善量 [dB]（平均と、試行を単位にしたブートストラップ 95% 区間）")
    for (rt, m), g in d.groupby(["rt60", "method"]):
        lo, hi = boot_ci(g["sdr_imp"])
        print(f"  RT60 {rt:.2f}s  {m:10s} {g['sdr_imp'].mean():6.2f}（{lo:6.2f}〜{hi:6.2f}）  n = {len(g)} 試行")

    ilr = d[d["method"].str.startswith("ILRMA")]
    tab = ilr.pivot_table(index="trial", columns="method", values="sdr_imp")   # 試行 × K（残響時間で平均）
    trials = [int(t) for t in tab.index]
    tune, test = trials[: len(trials) // 2], trials[len(trials) // 2:]
    print("\n[2] ILRMA の基底の数 K の選び方（同じ K の、選ぶのに使った試行での成績と、使っていない試行での成績）")
    best_tune, on_tune, on_test = select_k(tab, tune, test)
    print(f"  前半の試行 {tune} で選んだ K：{best_tune}")
    print(f"  【誤り】選ぶのに使った前半での平均 {on_tune:.2f} dB")
    print(f"  【正しい】使っていない後半 {test} での平均 {on_test:.2f} dB（かたより {on_tune - on_test:+.2f} dB。"
          "1回の分け方では負になることもある）")
    bias, chosen = [], []
    for _ in range(N_SPLIT):
        perm = list(RNG.permutation(trials))
        a, b = perm[: len(perm) // 2], perm[len(perm) // 2:]
        for s, t in ((a, b), (b, a)):                    # 前半・後半を入れ替えた分も使う
            k, x_s, x_t = select_k(tab, s, t)
            bias.append(x_s - x_t)
            chosen.append(k)
    bias = np.array(bias)
    ks = pd.Series(chosen).value_counts()
    print(f"  分け方を {len(bias)} 通りに変えると：かたよりの平均 {bias.mean():+.2f} dB"
          f"（負になった割合 {np.mean(bias < 0) * 100:.0f}%）")
    print("  選ばれた K の回数：" + "、".join(f"{k} {v} 回" for k, v in ks.items()))
    if len(ks) == 1:
        print("  （どの分け方でも同じ K が選ばれたので、かたよりの平均は 0 になる。選ぶことによるかたよりが生じない）")
    else:
        print("  選ぶのに使ったデータでの成績は、平均すると新しいデータでの成績より良く見える（楽観的なかたより）。")

    print(f"\n[3] {best_tune} と AuxIVA の比較（後半の試行、残響時間ごと。差 = ILRMA − AuxIVA）")
    n_rt = d["rt60"].nunique()
    if n_rt > 1:
        print(f"  （K は [2] で、前半の試行の成績を {n_rt} つの残響時間で平均して選んだもの。残響時間ごとには選び直さない）")
    else:
        print("  （K は [2] で、前半の試行の成績から選んだもの）")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    res = []
    for rt, g in d[d["trial"].isin(test)].groupby("rt60"):
        p = g.pivot_table(index="trial", columns="method", values="sdr_imp")
        diff = (p[best_tune] - p["AuxIVA"]).dropna()
        if len(diff) >= 2:
            lo, hi = boot_ci(diff)
            tlo, thi = t_ci(diff)
            # n = 5 では、どんな結果でも両側 p の最小値が 2/2^5 = 0.0625 なので、6 試行以上で検定する。
            # ホルム補正（2回）のあとで 0.05 を下回れるのは 7 試行以上（最小 p = 2/2^7 × 2 = 0.031）
            pv = stats.wilcoxon(diff).pvalue if len(diff) >= 6 else np.nan
            res.append((rt, diff, lo, hi, tlo, thi, pv))
        for _, row in p.iterrows():
            ax.plot([0, 1], [row["AuxIVA"], row[best_tune]], "-o", ms=3, alpha=0.6, color=f"C{list(d['rt60'].unique()).index(rt)}")
        ax.plot([], [], "-o", color=f"C{list(d['rt60'].unique()).index(rt)}", label=f"RT60 {rt:.2f}s")
    ok = [r[6] for r in res if not np.isnan(r[6])]
    adj = iter(holm(ok))
    for rt, diff, lo, hi, tlo, thi, pv in res:
        if np.isnan(pv):
            s = "（6試行未満なので検定は省略）"
        else:
            s = f"、ウィルコクソン p = {pv:.3g}（ホルム補正後 {next(adj):.3g}）"
        print(f"  RT60 {rt:.2f}s：差の平均 {diff.mean():+.2f} dB（ブートストラップ 95% 区間 {lo:+.2f}〜{hi:+.2f}、"
              f"t 区間 {tlo:+.2f}〜{thi:+.2f}）、{len(diff)} 試行{s}")
    print("  試行が少ないと、ブートストラップ区間は狭く出すぎやすい。t 区間と並べて見る。")
    ax.set_xticks([0, 1], ["AuxIVA", best_tune])
    ax.set_ylabel("SDR 改善量 [dB]")
    ax.set_title("同じ試行どうしを線で結んだ図（対応のある比較）")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig("bss_compare.png", dpi=130)
    print("\nbss_compare.png を保存しました")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit("使い方: uv run python bss_stats.py results.csv")
    main(sys.argv[1])

In [ ]:
import sys, os, re, subprocess, importlib, logging, warnings
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
warnings.filterwarnings("ignore", message="Glyph")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, Image, display


def run_py(*args):
    """ターミナルの uv run python … の代わり。このノートブックと同じ Python で動かし、出力を表示して文字列で返す"""
    env = dict(os.environ, PYTHONWARNINGS="ignore:Glyph")   # 文字が無いという警告を出さない（日本語のフォントがあるかは 0-1 のセルが表示する）
    p = subprocess.Popen([sys.executable, "-u", *map(str, args)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=env)
    out = []
    for line in p.stdout:
        if line.startswith("findfont:"):                  # 同じく、候補のうち見つからないフォントの知らせは表示しない
            continue
        print(line, end="")
        out.append(line)
    assert p.wait() == 0, f"{args[0]} がエラーで止まりました（上の出力を見る）"
    return "".join(out)


def show_png(path, width=900):
    """プログラムが保存した PNG をセルの下に表示する"""
    display(Image(filename=path, width=width))


def import_script(name):
    """7-6 のファイルを import する。ファイルの中の matplotlib.use("Agg") で、図がセルに出なくなるのを元に戻す"""
    backend = matplotlib.get_backend()
    mod = importlib.import_module(name)
    matplotlib.use(backend)
    matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    return mod


def listen_files(files, peak=0.3):
    """wav を並べて再生ボタンにする。一番大きい値が peak（0.3 ≒ −10 dBFS）になるよう全部を同じ倍率で小さくする。
    normalize=False を消さない（消すと最大値が 1 に引き上げられ、とても大きな音になる）"""
    xs = [sf.read(f) for f in files]
    gain = min(1.0, peak / max(np.max(np.abs(x)) for x, _ in xs))
    for f, (x, fs) in zip(files, xs):
        y = x * gain
        assert np.max(np.abs(y)) <= 0.32, "大きすぎる音は鳴らさない"
        print(f, f"（{gain:.2f} 倍にして再生）")
        display(Audio(y.T if y.ndim > 1 else y, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1, all_=False):
    """出力の文字列 text から、pattern の (…) の数を取り出す（all_=True ならすべて）"""
    if all_:
        return [float(m.group(i)) for m in re.finditer(pattern, text)]
    m = re.search(pattern, text)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i))


matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("準備ができました：", os.getcwd())

## 1. NMF を試す（7-3-1・7-3-2、手順 C の 4・5、`nmf.py`）

`run_ch7.sh` の NMF の 2 行を、録音の代わりに `--example trumpet`（librosa の例のトランペット）で実行します。KL は振幅、IS はパワーのスペクトログラムを分解します（合わせて十数秒）。

In [ ]:
out_kl = run_py("nmf.py", "--example", "trumpet", "--div", "kl", "--k", "8", "--iter", "300")
out_is = run_py("nmf.py", "--example", "trumpet", "--div", "is", "--k", "8", "--iter", "300")

In [ ]:
# 確かめ（教科書の手順の「確かめること」）
for name, out in [("KL", out_kl), ("IS", out_is)]:
    inc = num(r"増えた割合の最大 ([-+\d.e]+)", out)
    ok("増えていない：補助関数法のとおり" in out and inc < 1e-9,
       f"[C4] {name}：「増えていない：補助関数法のとおり」と表示された（増えた割合の最大 {inc:.2e}）")
files = ["nmf_trumpet_kl.png", "nmf_trumpet_is.png"] + [f"nmf_trumpet_{d}_k{i:02d}.wav" for d in ("kl", "is") for i in range(4)]
ok(all(os.path.exists(f) for f in files), "[C4] PNG 2 枚と、成分の wav 8 個（k00〜k03）ができた")

図で、基底（左下）が倍音の並んだ形、アクティベーション（右下）が音の出た時刻に立ち上がる形になっているかを見てください。

In [ ]:
show_png("nmf_trumpet_kl.png")
show_png("nmf_trumpet_is.png")

成分を聞き比べます（手順 C の 5、音量は小さく）。大きい成分の上位 4 つです。KL と IS で、小さい音（高い倍音・残響の尾）の扱いが違うかを聞いてください。4 つは同じ倍率で小さくしているので、成分どうしの大きさの違いはそのまま聞こえます。

In [ ]:
listen_files([f"nmf_trumpet_kl_k{i:02d}.wav" for i in range(4)])
listen_files([f"nmf_trumpet_is_k{i:02d}.wav" for i in range(4)])

`--k` を 4・16 に変えて、成分の分かれ方がどう変わるかを見ます（KL で。上の `nmf_trumpet_kl*` は上書きされます。合わせて 20 秒ほど）。

In [ ]:
for k in ["4", "16"]:
    out_k = run_py("nmf.py", "--example", "trumpet", "--div", "kl", "--k", k, "--iter", "300")
    ok("増えていない：補助関数法のとおり" in out_k, f"k = {k} でもコストが増えていない")
    show_png("nmf_trumpet_kl.png")

## 2. IS の更新の平方根を外して比べる（7-3-2、手順 C の 6）

教科書では `nmf.py` の IS の更新の `np.sqrt(...)` を外して実行し、終わったら元に戻します。ノートブックでは、`nmf.py` は書きかえずに、`nmf()` の中身を写して平方根だけを外した関数をこのセルの中で作り、同じスペクトログラムで 2 つのコストの減り方を並べます（数秒）。

In [ ]:
import inspect
nmf = import_script("nmf")
src = inspect.getsource(nmf.nmf)
src_nosqrt = (src.replace("def nmf(", "def nmf_nosqrt(")
                 .replace("W *= np.sqrt(((X / Y ** 2) @ H.T) / ((1 / Y) @ H.T + EPS))",
                          "W *= (((X / Y ** 2) @ H.T) / ((1 / Y) @ H.T + EPS))")
                 .replace("H *= np.sqrt((W.T @ (X / Y ** 2)) / (W.T @ (1 / Y) + EPS))",
                          "H *= ((W.T @ (X / Y ** 2)) / (W.T @ (1 / Y) + EPS))"))
assert src_nosqrt.count("np.sqrt") == 0, "平方根を外せていない（nmf.py の写し間違いを疑う）"
exec(src_nosqrt, vars(nmf))                     # nmf.py の中の名前（EPS・cost_is など）をそのまま使う
nmf_nosqrt = nmf.nmf_nosqrt

import librosa
y, sr = librosa.load(librosa.ex("trumpet"), sr=16000, mono=True)   # nmf.py の既定と同じ（16 kHz、窓 1024、ずらし 256）
X = np.abs(librosa.stft(y, n_fft=1024, hop_length=256)) ** 2      # IS はパワー
_, _, h_sqrt = nmf.nmf(X, 8, "is", 300, 0)
_, _, h_nosq = nmf_nosqrt(X, 8, "is", 300, 0)
for name, h in [("平方根あり（補助関数法）", h_sqrt), ("平方根なし", h_nosq)]:
    inc = np.max(np.diff(h) / np.abs(h[:-1]))
    print(f"{name:14s} 最初 {h[0]:.4g} → 10 回目 {h[9]:.4g} → 最後 {h[-1]:.4g}、1 回で増えた割合の最大 {inc:.2e}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(h_sqrt, label="平方根あり")
ax.semilogy(h_nosq, label="平方根なし")
ax.set(xlabel="反復回数", ylabel="IS ダイバージェンス", title="IS の更新：平方根のあり・なし")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

In [ ]:
# 確かめ（教科書の手順の「確かめること」）
inc_sqrt = np.max(np.diff(h_sqrt) / np.abs(h_sqrt[:-1]))
inc_nosq = np.max(np.diff(h_nosq) / np.abs(h_nosq[:-1]))
ok(inc_sqrt < 1e-9, f"[C6] 平方根あり：コストが増えていない（{inc_sqrt:.2e}）")
ok(inc_nosq < 1e-9, f"[C6] 平方根なしでも、コストは増えない（{inc_nosq:.2e}。IV-4-3-8）")
ok(h_nosq[-1] < h_nosq[0] and h_sqrt[-1] < h_sqrt[0], "[C6] どちらも最初より最後のほうが小さい")

1 回あたりの減り方の違い（はじめの数回で、どちらが大きく減るか）をノートに書いてください。

## 3. 2 チャネルにして分離する（7-3-3・7-3-4、手順 D の 7・8、`bss_experiment.py`）

`run_ch7.sh` の「録音がなければ」の行（`libri1` と `trumpet`、`--out results_example.csv`）を実行します。**ノートブックの既定は試しの設定**（`--trials 4`：2 つの残響時間 × 4 試行 × 4 手法。執筆時の Mac で約 2 分）です。教科書の本番（`--trials 14`）にするときは、`TRIALS = 4` の行をコメントの行に取りかえてください。

`bss_experiment.py` は、結果のファイルがすでにあると上書きしないように止まります。ノートブックではセルを何度でも実行できるように、実行の前に `results_example.csv` を消しています（教科書の本番の結果を残したいときは、先に名前を変えておいてください）。

In [ ]:
TRIALS = 4      # 試しの設定。教科書の本番は次の行の # を外す（28 試行。Mac で約 7 分、Colab ではその数倍）
# TRIALS = 14
OUT = "results_example.csv"
if os.path.exists(OUT):
    os.remove(OUT)                  # ノートブック版：何度でも実行できるように（上の説明）
out_exp = run_py("bss_experiment.py", "--rt60", "0.15", "0.3", "--trials", TRIALS, "--out", OUT)

In [ ]:
# 確かめ（教科書の手順の「確かめること」）
d = pd.read_csv(OUT)
ok(len(d) == 2 * TRIALS * 4 and set(d["method"]) == {"AuxIVA", "ILRMA_K2", "ILRMA_K4", "ILRMA_K10"},
   f"[D7] {OUT} に 2 残響時間 × {TRIALS} 試行 × 4 手法 = {2 * TRIALS * 4} 行")
ok(len(re.findall(r"^RT60 .* SDR .* SIR .* SAR ", out_exp, re.M)) == len(d), "[D7] 1 行ごとに SDR・SIR・SAR が表示された")
ok(np.all(np.isfinite(d[["sdr", "sir", "sar", "sdr_imp"]].to_numpy())), "[D7] SDR・SIR・SAR が有限の値")
m = d.groupby("rt60")["sdr"].mean()
ok(m[0.15] > m[0.3], f"[D7] 残響時間 0.15 s のほうが 0.3 s より SDR が高い（手法と試行の平均 {m[0.15]:.2f} dB と {m[0.3]:.2f} dB）")
g = d[d["rt60"] == 0.15].groupby("method")["sdr_imp"].mean()
best = g.drop("AuxIVA").idxmax()
ok(g[best] > g["AuxIVA"], f"[D7] 0.15 s では ILRMA（一番よい {best}：{g[best]:.2f} dB）が AuxIVA（{g['AuxIVA']:.2f} dB）より改善が大きい")
wavs = [f"sep/rt{r}_{n}" for r in (150, 300) for n in ["mix_mic0.wav"] + [f"{m_}_y{i}.wav" for m_ in ("AuxIVA", "ILRMA_K2", "ILRMA_K4", "ILRMA_K10") for i in (1, 2)]]
ok(all(os.path.exists(f) for f in wavs), f"[D7] sep/ に混合音と分離音の wav が {len(wavs)} 個できた")

分離音を聞き比べます（手順 D の 8、音量は小さく）。残響時間 0.15 s の試行 0 の、混合音（基準マイク）→ AuxIVA → ILRMA（K = 2）です。y1・y2 のどちらが話し声になるかは決まっていません。分離音では、もう一方の音源が小さくなっているかを聞いてください。

In [ ]:
listen_files(["sep/rt150_mix_mic0.wav", "sep/rt150_AuxIVA_y1.wav", "sep/rt150_AuxIVA_y2.wav",
              "sep/rt150_ILRMA_K2_y1.wav", "sep/rt150_ILRMA_K2_y2.wav"])

残響時間 0.3 s では、分離が弱くなる、音がこもる、などの違いが聞こえるかを比べます。

In [ ]:
listen_files(["sep/rt300_mix_mic0.wav", "sep/rt300_AuxIVA_y1.wav", "sep/rt300_ILRMA_K2_y1.wav"])

## 4. コストが増えないことを確かめる（7-3-3、手順 D の 9、`bss.py`）

教科書では Python の対話モードで行う手順です。`X` は `bss_experiment.py` の中と同じように作ります（残響時間 0.15 s の試行 0 と同じ部屋。乱数の種は `1000 × 0 + 150`）。数十秒かかります。

In [ ]:
import argparse
bss = import_script("bss")
be = import_script("bss_experiment")
a = argparse.Namespace(src1=None, src2=None, dur=6.0)            # bss_experiment.py の既定（libri1 と trumpet、6 秒）
src = be.load_sources(a, 16000)
mix, ref = be.simulate(src, 16000, 0.15, np.random.default_rng(1000 * 0 + int(0.15 * 1000)))
X = np.stack([librosa.stft(m, n_fft=be.NFFT, hop_length=be.HOP) for m in mix], axis=-1)   # (F, T, M)
print("X の形（F, T, M）：", X.shape)
Y, W, hist = bss.ilrma(X, 2, 50, return_cost=True)
Ya, Wa, hist_a = bss.auxiva(X, 50, return_cost=True)
for name, h in [("ILRMA（K=2）", hist), ("AuxIVA", hist_a)]:
    print(f"{name:12s} 最初 {h[0]:.6g} → 最後 {h[-1]:.6g}、np.max(np.diff(hist)) = {np.max(np.diff(h)):.3g}")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (name, h) in zip(axs, [("ILRMA（K=2）", hist), ("AuxIVA", hist_a)]):
    ax.plot(h)
    ax.set(xlabel="反復回数", ylabel="コスト（負の対数尤度）", title=name)
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# 確かめ（教科書の手順の「確かめること」）
for name, h in [("ILRMA", hist), ("AuxIVA", hist_a)]:
    ok(np.max(np.diff(h)) <= 1e-9 * np.max(np.abs(h)) and h[-1] < h[0],
       f"[D9] {name}：np.max(np.diff(hist)) = {np.max(np.diff(h)):.3g} が 0 以下（数値誤差の範囲）で、コストが減った")

## 5. 統計的に比べる（7-3-4、手順 E の 10、`bss_stats.py`）

3 節の `results_example.csv` を使います（数秒）。図 `bss_compare.png` で、右上がり（ILRMA のほうがよい）の線が何本あり、右下がりの線がどの試行かを見てください（試行が少ないと、本文 7-7 の例と数が違います）。4 試行の試しの設定では、[3] の比較は後半の 2 試行だけなので、検定は省略されます。

In [ ]:
out_st = run_py("bss_stats.py", OUT)
show_png("bss_compare.png", width=650)

In [ ]:
# 確かめ（教科書の手順の「確かめること」）
n_test = TRIALS - TRIALS // 2
l1 = re.findall(r"^  RT60 [\d.]+s  (\S+)\s+[-\d.]+（\s*[-\d.]+〜\s*[-\d.]+）  n = (\d+) 試行", out_st, re.M)
ok(len(l1) == 8 and all(int(n) == TRIALS for _, n in l1), f"[E10][1] 手法 × 残響時間の 8 行に、平均と区間（n = {TRIALS} 試行）")
k_sel = re.search(r"前半の試行 .* で選んだ K：(ILRMA_K\d+)", out_st).group(1)
ok(k_sel in {"ILRMA_K2", "ILRMA_K4", "ILRMA_K10"} and "【誤り】" in out_st and "【正しい】" in out_st,
   f"[E10][2] 前半で選んだ K（{k_sel}）の【誤り】（前半での成績）と【正しい】（後半での成績）が表示された")
bias = num(r"かたよりの平均 ([-+\d.]+) dB", out_st)
ok("400 通り" in out_st and ("同じ K が選ばれた" not in out_st or bias == 0),
   f"[E10][2] 分け方を 400 通りに変えたかたよりの平均 {bias:+.2f} dB（どの分け方でも同じ K なら 0）")
l3 = re.findall(r"^  RT60 [\d.]+s：差の平均 .*、(\d+) 試行(.*)$", out_st, re.M)
ok(len(l3) == 2 and all(int(n) == n_test for n, _ in l3), f"[E10][3] 2 つの残響時間で、後半の {n_test} 試行の差の平均と区間")
ok(all(("検定は省略" in s) == (n_test < 6) for _, s in l3),
   "[E10][3] 後半が 6 試行未満なら検定は省略、6 試行以上ならウィルコクソンの p とホルム補正後の p")
ok(os.path.exists("bss_compare.png"), "[E10] bss_compare.png ができた")

7-3-4 と 7-7 に書いてある「試行が少ないと検定できない」理由を、`bss_stats.py` の `holm()` と scipy のウィルコクソンの符号順位検定で、数で確かめます（結果の CSV には関係しない計算です）。

In [ ]:
from scipy import stats
bs = import_script("bss_stats")
for n in [5, 6, 7]:
    print(f"{n} 試行の差がすべて正のときの両側 p：{stats.wilcoxon(np.arange(1, n + 1)).pvalue:.4f}（2/2^{n} = {2 / 2 ** n:.4f}）")
print("ホルム補正の例：p = [0.015625, 0.2] →", bs.holm([0.015625, 0.2]))

In [ ]:
# 確かめ（教科書の手順の「確かめること」）
ok(abs(stats.wilcoxon(np.arange(1, 6)).pvalue - 0.0625) < 1e-12, "5 試行では、すべて正でも両側 p は 0.0625（0.05 を下回らない）")
ok(abs(stats.wilcoxon(np.arange(1, 8)).pvalue - 0.015625) < 1e-12, "7 試行すべてで正のときの最小の p は 0.015625（7-3-4 の最小の p）")
ok(np.allclose(bs.holm([0.015625, 0.2]), [0.03125, 0.2]), "ホルム補正（2 回）で 0.015625 → 0.031（7-3-4 のホルム補正後の最小値）")

## おわりに

- NMF（KL と IS）の乗法更新でコストが増えないこと、IS の平方根を外しても増えないこと、AuxIVA と ILRMA の分離、そのコストが増えないこと、試行を単位にした統計の比べ方を、数値と図と音で確かめました。
- 3 節を教科書の本番（`--trials 14`）で行うと、5 節の [3] でウィルコクソンの p とホルム補正後の p が表示され、教科書の 7-7 の結果の例と比べられます。
- このノートブックに入れていないもの：手順 B の録音（`record.py`。Mac のマイクが要る）と、手順 E の 11（論文を読む。コードがない。7-3-5 の表とコツに従って、ILRMA の論文と `bss.py` を照らし合わせる）。